In [1]:
library('dplyr')
library('MatchIt')
library('lmtest')
library('sandwich')
library('cem')
library('lme4')
library(ggplot2)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loading required package: zoo


Attaching package: ‘zoo’


The following objects are masked from ‘package:base’:

    as.Date, as.Date.numeric


Loading required package: tcltk

Warning message:
“no DISPLAY variable so Tk is not available”
Loading required package: lattice


How to use CEM? Type vignette("cem")


Loading required package: Matrix



In [2]:
data_path='../../data/dropbox/LaborDivision/Data/deriveddata/matching/author_meta.csv'

In [13]:
df=read.csv(data_path)
head(df)

,cluster_id,dis,gender,yfp,npub,F,L,M,intcnt,Code,ingroup
,<int>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>
1,20009795,51,F,2008,5,2,0,0,2,JPN,H
2,29411586,51,M,2000,74,3,0,5,8,JPN,H
3,1221298,51,M,1980,21,0,0,2,2,JPN,H
4,38679117,51,F,1980,56,0,2,2,4,JPN,H
5,22736843,26,M,2008,1,0,0,1,1,JPN,H
6,21683477,26,M,2008,1,0,0,1,1,JPN,H


In [14]:
df=df[df$npub<100,]
df$gender=as.factor(df$gender)
df$dis=as.factor(df$dis)

In [15]:
yearcut=c(1980,1989.5,seq(1999.5,2020.5))
pubcut=seq(0,101,by=5)

In [6]:
cntrys=c('BGD','ZAF','THA','QAT','CHL','KEN','KOR')
cols=c('att','attp','low','upper')
dff=data.frame(matrix(nrow=length(cntrys),ncol=length(cols)))
colnames(dff)=cols
rownames(dff)=cntrys

dfl=data.frame(matrix(nrow=length(cntrys),ncol=length(cols)))
colnames(dfl)=cols
rownames(dfl)=cntrys

dfm=data.frame(matrix(nrow=length(cntrys),ncol=length(cols)))
colnames(dfm)=cols
rownames(dfm)=cntrys

In [26]:
for (x in cntrys){
    df1=df[df$Code %in% c('USA',x),]
    df1$treat=ifelse(df1$Code==x,1,0)
    
    matching1 <- matchit(treat ~ yfp + gender+npub+intcnt+dis, 
                         data = df1, 
                         cutpoints=list(yfp=yearcut,npub=pubcut,intcnt=pubcut),
                         method='cem',
                         estimand = 'ATT')
    
    matched_df1 <- match.data(matching1)
    
    modelF <- lm(F ~ treat, data = matched_df1,weights=weights)
    att = summary(modelF)$coefficients['treat','Estimate']
    attp=summary(modelF)$coefficients['treat','Pr(>|t|)']
    confdf = data.frame(confint(modelF))
    colnames(confdf)<-list("l","u")
    confl=confdf['treat',]$l
    confu=confdf['treat',]$u
    dff[x,'att']=att
    dff[x,'attp']=attp
    dff[x,'low']=confl
    dff[x,'upper']=confu
    
    modell <- lm(L ~ treat, data = matched_df1,weights=weights)
    att = summary(modell)$coefficients['treat','Estimate']
    attp=summary(modell)$coefficients['treat','Pr(>|t|)']
    confdf = data.frame(confint(modell))
    colnames(confdf)<-list("l","u")
    confl=confdf['treat',]$l
    confu=confdf['treat',]$u
    dfl[x,'att']=att
    dfl[x,'attp']=attp
    dfl[x,'low']=confl
    dfl[x,'upper']=confu
    
    modelm <- lm(M ~ treat, data = matched_df1,weights=weights)
    att = summary(modelm)$coefficients['treat','Estimate']
    attp=summary(modelm)$coefficients['treat','Pr(>|t|)']
    confdf = data.frame(confint(modelm))
    colnames(confdf)<-list("l","u")
    confl=confdf['treat',]$l
    confu=confdf['treat',]$u
    dfm[x,'att']=att
    dfm[x,'attp']=attp
    dfm[x,'low']=confl
    dfm[x,'upper']=confu
}

In [33]:
write.csv(dff,'first1.csv')

In [34]:
write.csv(dfl,'last1.csv')

In [35]:
write.csv(dfm,'middle1.csv')

### investigate the difference between matched units and dropped units

In [18]:
diff_df=data.frame()
cntrys=c('BGD','ZAF','THA','QAT','CHL','KEN','KOR')
for (x in cntrys){

    df1=df[df$Code %in% c('USA',x),]
    df1$treat=ifelse(df1$Code==x,1,0)

    matching1 <- matchit(treat ~ yfp + gender+npub+intcnt+dis, 
                         data = df1, 
                         cutpoints=list(yfp=yearcut,npub=pubcut,intcnt=pubcut),
                         method='cem',
                         estimand = 'ATT')

    matched_df1 <- match.data(matching1, drop.unmatched=FALSE) #keep unmatched records
    usa_df=subset(matched_df1, Code=='USA')
    usa_df$match=ifelse(is.na(usa_df$subclass),'U','M') #U-unmatched, M-matched
    res_df=usa_df%>% group_by(match) %>% summarise(pub=mean(npub),
                                             int=mean(intcnt),
                                                  first=mean(F),
                                                  last=mean(L),
                                                  year=mean(yfp))
    res_df$Code=x
    diff_df=rbind(diff_df,res_df)
}

In [19]:
diff_df

match,pub,int,first,last,year,Code
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
M,2.035031,1.313779,0.1655217,0.08030566,2014.964,BGD
U,15.436533,5.189663,0.6570649,0.65220630,2005.978,BGD
M,3.103808,1.598677,0.2086004,0.11699246,2013.268,ZAF
U,20.142084,6.589119,0.8234799,0.86722343,2004.383,ZAF
M,2.814123,1.460251,0.1931400,0.09303321,2013.670,THA
U,19.511834,6.481331,0.8079122,0.85553367,2004.351,THA
M,2.132563,1.354241,0.1621553,0.08224646,2015.548,QAT
U,13.751899,4.695822,0.5990280,0.58151604,2006.756,QAT
M,3.226361,1.637330,0.2178932,0.11923391,2013.124,CHL


In [59]:
matching1 <- matchit(treat ~ yfp + gender+npub+tot_int, 
                     data = df1, 
                     cutpoints=list(yfp=yearcut,npub=pubcut,tot_int=pubcut),
                     k2k = TRUE, k2k.method = "mahalanobis",
                     estimand = 'ATT')

In [1]:
summary(matching1, un=FALSE)

ERROR: Error in summary(matching1, un = FALSE): object 'matching1' not found


In [77]:
matched_df1 <- match.data(matching1)

In [65]:
model2 <- lm(F ~ treat, data = matched_df1,weights=weights)

In [79]:
summary(model2)


Call:
lm(formula = F ~ treat, data = matched_df1, weights = weights)

Residuals:
   Min     1Q Median     3Q    Max 
-0.784 -0.784 -0.399  0.216 32.216 

Coefficients:
            Estimate Std. Error t value Pr(>|t|)    
(Intercept) 0.398926   0.009659    41.3   <2e-16 ***
treat       0.385226   0.013659    28.2   <2e-16 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 1.095 on 25692 degrees of freedom
Multiple R-squared:  0.03003,	Adjusted R-squared:  0.02999 
F-statistic: 795.4 on 1 and 25692 DF,  p-value: < 2.2e-16


In [51]:
att = summary(model2)$coefficients['treat','Estimate']
attp=summary(model2)$coefficients['treat','Pr(>|t|)']
confdf = data.frame(confint(model2))
colnames(confdf)<-list("l","u")
confl=confdf['treat',]$l
confu=confdf['treat',]$u

In [2]:
summary(model2)

ERROR: Error in summary(model2): object 'model2' not found


In [16]:
summary(model2)


Call:
lm(formula = L ~ treat, data = matched_df1, weights = weights)

Weighted Residuals:
   Min     1Q Median     3Q    Max 
-2.042 -0.300 -0.300 -0.086 47.700 

Coefficients:
            Estimate Std. Error t value Pr(>|t|)    
(Intercept)  0.18798    0.02255   8.337  < 2e-16 ***
treat        0.11193    0.02477   4.518 6.29e-06 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 1.164 on 15508 degrees of freedom
Multiple R-squared:  0.001315,	Adjusted R-squared:  0.00125 
F-statistic: 20.41 on 1 and 15508 DF,  p-value: 6.288e-06


In [69]:
imbalance(group=df1$treat,data=df1[vars])


Multivariate Imbalance Measure: L1=0.418
Percentage of local common support: LCS=2.3%

Univariate Imbalance Measures:

        statistic   type           L1 min 25% 50% 75%  max
yfp     -3.392404 (diff) 4.884981e-15   0  -4  -3  -1    0
gender  37.306868 (Chi2) 2.616935e-02  NA  NA  NA  NA   NA
npub     7.766501 (diff) 1.868292e-01   0   0   1   7 1556
tot_int  1.513756 (diff) 1.079595e-01   0   0   0   0 1288

In [94]:
lv <- unique(na.omit(df1$treat))

In [95]:
lv

[1] 0 1

In [129]:
mat1=cem(treatment='treat',data=df1,drop=c('cluster_id','Code','ingroup','F','L','M'),
         cutpoints=list(yfp=yearcut,npub=pubcut,tot_int=pubcut),keep.all=TRUE)



Using 'treat'='1' as baseline group


In [130]:
mat1

               G0    G1
All       1149501 12867
Matched   1041990 12867
Unmatched  107511     0


In [131]:
mat2=k2k(mat1,data=df1,'euclidean',1)

In [77]:
est=att(mat1,F~treat,data=df1, model='linear-RE')

In [136]:
matching0 <- matchit(treat ~ yfp + gender+npub+tot_int, data = df1, 
                     method = NULL)
summary(matching0, un=FALSE)


Call:
matchit(formula = treat ~ yfp + gender + npub + tot_int, data = df1, 
    method = NULL)

Summary of Balance for All Data:
         Means Treated Means Control Std. Mean Diff. Var. Ratio eCDF Mean
distance        0.0137        0.0110          0.8382     0.3134    0.0822
yfp          2012.9496     2009.5572          0.5206     0.4503    0.0827
genderF         0.3950        0.3689          0.0535          .    0.0262
genderM         0.6050        0.6311         -0.0535          .    0.0262
npub            4.7867       12.5532         -0.6673     0.1315    0.0088
tot_int         2.8726        4.3864         -0.2420     0.1937    0.0031
         eCDF Max
distance   0.2332
yfp        0.1490
genderF    0.0262
genderM    0.0262
npub       0.1864
tot_int    0.0548


Sample Sizes:
          Control Treated
All       1149501   12867
Matched   1149501   12867
Unmatched       0       0
Discarded       0       0


In [152]:
mean(matched_df1[matched_df1$Code=='USA',]$L)

[1] 0.2597342

In [153]:
mean(matched_df1[matched_df1$Code=='DZA',]$L)

[1] 0.329292

In [154]:
write.csv(matched_df1,'test.csv')

In [145]:
summary(model2)


Call:
lm(formula = F ~ treat, data = matched_df1)

Residuals:
   Min     1Q Median     3Q    Max 
-0.793 -0.793 -0.408  0.207 57.207 

Coefficients:
            Estimate Std. Error t value Pr(>|t|)    
(Intercept)  0.40787    0.01028   39.66   <2e-16 ***
treat        0.38509    0.01454   26.48   <2e-16 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 1.166 on 25732 degrees of freedom
Multiple R-squared:  0.02653,	Adjusted R-squared:  0.02649 
F-statistic: 701.3 on 1 and 25732 DF,  p-value: < 2.2e-16


In [116]:
summary(model2)


Call:
lm(formula = L ~ treat, data = matched_df1[matched_df1$subclass == 
    112, ])

Residuals:
    Min      1Q  Median      3Q     Max 
-5.8765 -3.8765 -0.8765  3.1235 22.1235 

Coefficients:
            Estimate Std. Error t value Pr(>|t|)    
(Intercept)    5.877      0.121  48.572   <2e-16 ***
treat          4.790      2.827   1.694   0.0904 .  
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 4.892 on 1636 degrees of freedom
Multiple R-squared:  0.001752,	Adjusted R-squared:  0.001142 
F-statistic: 2.871 on 1 and 1636 DF,  p-value: 0.09037
